# D-01 EDA - District Interstate Flows (Census 2011)

This notebook builds the district-level interstate inflow CSV used by the React district migration page.

Outputs:
- `data-cleaning/district_interstate_flows.csv`
- `react-app/public/district_interstate_flows.csv`


In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
SOURCE_DIR = ROOT / 'D-01-ALL'
OUTPUT_CLEAN = ROOT / 'district_interstate_flows.csv'
OUTPUT_PUBLIC = ROOT.parent / 'react-app' / 'public' / 'district_interstate_flows.csv'

STATE_NAMES = {
    'JAMMU & KASHMIR','HIMACHAL PRADESH','PUNJAB','CHANDIGARH','UTTARAKHAND',
    'HARYANA','NCT OF DELHI','RAJASTHAN','UTTAR PRADESH','BIHAR','SIKKIM',
    'ARUNACHAL PRADESH','NAGALAND','MANIPUR','MIZORAM','TRIPURA','MEGHALAYA',
    'ASSAM','WEST BENGAL','JHARKHAND','ODISHA','CHHATTISGARH','MADHYA PRADESH',
    'GUJARAT','DAMAN & DIU','DADRA & NAGAR HAVELI','MAHARASHTRA','ANDHRA PRADESH',
    'KARNATAKA','GOA','LAKSHADWEEP','KERALA','TAMIL NADU','PUDUCHERRY',
    'ANDAMAN & NICOBAR ISLANDS','TELANGANA'
}

COLUMNS = [
    'TableName','StateCode','DistrictCode','AreaName','BirthPlace',
    'TotalPersons','TotalMales','TotalFemales','RuralPersons','RuralMales','RuralFemales',
    'UrbanPersons','UrbanMales','UrbanFemales'
]

def normalize_name(name):
    if name is None or (isinstance(name, float) and pd.isna(name)):
        return ''
    value = str(name).upper().strip()
    value = value.replace('UNION TERRITORY - ', '')
    value = value.replace('STATE - ', '')
    value = value.replace('UT - ', '')
    mapping = {
        'DELHI': 'NCT OF DELHI',
        'ORISSA': 'ODISHA',
        'TELENGANA': 'TELANGANA',
        'UTTARANCHAL': 'UTTARAKHAND',
        'JAMMU AND KASHMIR': 'JAMMU & KASHMIR',
        'ANDAMAN AND NICOBAR': 'ANDAMAN & NICOBAR ISLANDS',
        'ANDAMAN AND NICOBAR ISLANDS': 'ANDAMAN & NICOBAR ISLANDS',
        'ANDAMAN & NICOBAR': 'ANDAMAN & NICOBAR ISLANDS',
        'DADRA AND NAGAR HAVELI': 'DADRA & NAGAR HAVELI',
        'DAMAN AND DIU': 'DAMAN & DIU'
    }
    return mapping.get(value, value)

def to_int(value):
    try:
        if pd.isna(value):
            return 0
    except TypeError:
        pass
    try:
        return int(float(value))
    except (TypeError, ValueError):
        return 0

TELANGANA_DISTRICTS = {
    'ADILABAD','NIZAMABAD','KARIMNAGAR','MEDAK','WARANGAL',
    'RANGAREDDI','RANGAREDDY','HYDERABAD','NALGONDA','KHAMMAM','MAHBUBNAGAR'
}


In [ ]:
rows = []

for workbook in sorted(SOURCE_DIR.glob('*.XLSX')):
    frame = pd.read_excel(workbook, sheet_name='D-01', header=1)
    frame.columns = COLUMNS
    frame = frame.iloc[3:].copy()
    workbook_state = normalize_name(frame.iloc[0]['AreaName'])

    for _, row in frame.iterrows():
        district_code = to_int(row['DistrictCode'])
        if district_code == 0:
            continue

        origin_state = normalize_name(row['BirthPlace'])
        if origin_state not in STATE_NAMES:
            continue

        district_name = str(row['AreaName']).strip()
        if not district_name or district_name.upper().startswith('STATE -') or district_name.upper().startswith('UNION TERRITORY -'):
            continue

        count = to_int(row['TotalPersons'])
        if count <= 0:
            continue

        display_state = 'TELANGANA' if district_name.upper() in TELANGANA_DISTRICTS else workbook_state

        rows.append({
            'state': display_state,
            'district': district_name,
            'districtCode': district_code,
            'origin': origin_state,
            'count': count,
            'male': to_int(row['TotalMales']),
            'female': to_int(row['TotalFemales']),
            'rural': to_int(row['RuralPersons']),
            'urban': to_int(row['UrbanPersons']),
        })

df = pd.DataFrame(rows).sort_values(['state', 'district', 'count'], ascending=[True, True, False]).reset_index(drop=True)
df.to_csv(OUTPUT_CLEAN, index=False)
df.to_csv(OUTPUT_PUBLIC, index=False)
print(f"Rows: {len(df):,}")
print(f"Rows: {len(df):,}")
print(f"States: {df['state'].nunique():,}")
print(f"Districts: {df[['state','district']].drop_duplicates().shape[0]:,}")
print(f"Clean output: {OUTPUT_CLEAN}")
print(f"Public output: {OUTPUT_PUBLIC}")
df.head(10)
